# Lesson 2: Neural Networks for Energy Classification

## Neural Network with Synthetic Energy Data

This notebook presents a structured neural-network classification example within the smart-city and energy-management context.

The lesson uses synthetic energy-style data and trains a multi-layer neural network to predict:

- Class `1`: high energy demand
- Class `0`: not high energy demand

This notebook extends the perceptron lesson by introducing hidden layers, scaled inputs, and nonlinear classification.

## Step 1: Import the Required Tools

This step imports the libraries required for synthetic data generation, preprocessing, neural-network training, and evaluation.

- `random` helps us generate synthetic values.
- `dataclass` gives us a clean structure for each sample.
- `numpy` is useful for numerical arrays.
- `sklearn` provides scaling, the neural-network model, and evaluation metrics.

In [1]:
import random
from dataclasses import dataclass

import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

## Step 2: Create a `Sample` Class

This class represents one observation in our dataset.

Each sample includes:

- `features`: the input values used by the neural network
- `label`: the correct target class

In [2]:
@dataclass
class Sample:
    # Step 2.1: Store the feature vector for one observation.
    features: list[float]

    # Step 2.2: Store the expected class label.
    label: int

## Step 3: Define Helper Functions for Feature Preparation

We normalize the raw synthetic values so the input variables are easier for the model to handle.

This keeps the feature ranges more consistent before standard scaling.

In [3]:
def normalize_temperature(temp_c: float) -> float:
    """
    Step 3.1:
    Normalize temperature to a smaller range.
    """
    return (temp_c - 10.0) / 25.0


def normalize_occupancy(occupancy: int) -> float:
    """
    Step 3.2:
    Convert occupancy into a value between 0 and 1.
    """
    return occupancy / 100.0

## Step 4: Generate One Synthetic Energy Sample

This function creates one artificial sample that mimics simple energy-demand behavior.

The synthetic variables are:

- outdoor temperature
- occupancy level
- hour of day
- weekday indicator

We then derive a target label based on a noisy energy score.

In [4]:
def generate_sample() -> Sample:
    """
    Step 4:
    Create one synthetic sample for the neural-network lesson.
    """
    # Step 4.1: Generate raw synthetic values.
    temperature = random.uniform(8.0, 38.0)
    occupancy = random.randint(5, 100)
    hour = random.randint(0, 23)
    is_weekday = random.randint(0, 1)

    # Step 4.2: Derive a business-hours feature.
    business_hours = 1 if 8 <= hour <= 18 else 0

    # Step 4.3: Create a synthetic energy score.
    # The rule includes a small nonlinear term so that a neural network
    # is a more natural next example after the perceptron lesson.
    interaction_term = normalize_temperature(temperature) * normalize_occupancy(occupancy)

    energy_score = (
        0.7 * normalize_temperature(temperature)
        + 1.0 * normalize_occupancy(occupancy)
        + 0.5 * business_hours
        + 0.3 * is_weekday
        + 0.9 * interaction_term
        + random.uniform(-0.20, 0.20)
    )

    # Step 4.4: Convert the score into a binary class label.
    high_demand = 1 if energy_score > 1.45 else 0

    # Step 4.5: Build the neural-network input vector.
    features = [
        normalize_temperature(temperature),
        normalize_occupancy(occupancy),
        float(business_hours),
        float(is_weekday),
        interaction_term,
    ]

    return Sample(features=features, label=high_demand)

## Step 5: Generate a Full Dataset

A machine learning model needs many examples.

This function repeats the sample-generation process and returns a dataset.

In [5]:
def generate_dataset(size: int) -> list[Sample]:
    """
    Step 5:
    Create a dataset with the requested number of synthetic samples.
    """
    return [generate_sample() for _ in range(size)]

## Step 6: Convert Samples into Arrays

Scikit-learn models expect array-like inputs.

This function converts our `Sample` objects into:

- `X`: feature matrix
- `y`: target vector

In [6]:
def samples_to_arrays(dataset: list[Sample]) -> tuple[np.ndarray, np.ndarray]:
    """
    Step 6:
    Convert a list of Sample objects into X and y arrays.
    """
    X = np.array([sample.features for sample in dataset], dtype=float)
    y = np.array([sample.label for sample in dataset], dtype=int)
    return X, y

## Step 7: Prepare Evaluation Functions

We want a full evaluation, not only accuracy.

The next function computes:

- accuracy
- precision
- recall
- F1-score
- confusion matrix
- classification report

In [7]:
def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Step 7:
    Compute a complete set of evaluation metrics.
    """
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "classification_report": classification_report(y_true, y_pred, zero_division=0),
    }

## Step 8: Display a Few Samples in a Readable Format

This helper function is only for readable output when we inspect predictions.

In [8]:
def pretty_feature_row(features: np.ndarray) -> str:
    """
    Step 8:
    Convert one feature vector into a readable text line.
    """
    return (
        f"temp={features[0]:.2f}, "
        f"occupancy={features[1]:.2f}, "
        f"business_hours={features[2]:.0f}, "
        f"weekday={features[3]:.0f}, "
        f"interaction={features[4]:.2f}"
    )

## Step 9: Create the Training and Test Data

We generate two datasets:

- a training set for learning
- a test set for final evaluation

We also fix the random seed so the results are reproducible.

In [9]:
# Step 9.1: Fix the random seed for reproducibility.
random.seed(42)
np.random.seed(42)

# Step 9.2: Generate the datasets.
train_data = generate_dataset(400)
test_data = generate_dataset(120)

# Step 9.3: Convert the datasets into arrays.
X_train, y_train = samples_to_arrays(train_data)
X_test, y_test = samples_to_arrays(test_data)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Number of features: {X_train.shape[1]}")

Training samples: 400
Test samples: 120
Number of features: 5


## Step 10: Standardize the Input Features

Neural networks generally benefit from scaled inputs.

We fit the scaler on the training data only, then transform both training and test sets.

In [10]:
# Step 10.1: Create the scaler.
scaler = StandardScaler()

# Step 10.2: Fit on training data and transform both datasets.
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

Feature scaling completed.


## Step 11: Build the Neural Network Model

We now create a multi-layer perceptron classifier.

This model has:

- an input layer
- two hidden layers
- an output layer for binary classification

The hidden layers allow the network to learn more complex patterns than a single neuron.

In [11]:
# Step 11.1: Create the neural-network classifier.
model = MLPClassifier(
    hidden_layer_sizes=(8, 4),
    activation="relu",
    solver="adam",
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42,
)

print(model)

MLPClassifier(hidden_layer_sizes=(8, 4), learning_rate_init=0.01, max_iter=1000,
              random_state=42)


## Step 12: Train the Neural Network

Now we fit the model on the scaled training data.

During training, the neural network adjusts its internal weights to reduce prediction error.

In [12]:
# Step 12: Train the neural network.
model.fit(X_train_scaled, y_train)

print("Neural-network training completed.")
print(f"Training iterations used: {model.n_iter_}")
print(f"Final training loss: {model.loss_:.4f}")

Neural-network training completed.
Training iterations used: 275
Final training loss: 0.0678


## Step 13: Evaluate the Model

We now generate predictions for both datasets and calculate a full set of evaluation scores.

In [13]:
# Step 13.1: Generate predictions.
train_predictions = model.predict(X_train_scaled)
test_predictions = model.predict(X_test_scaled)

# Step 13.2: Evaluate both datasets.
train_results = evaluate_predictions(y_train, train_predictions)
test_results = evaluate_predictions(y_test, test_predictions)

# Step 13.3: Print the main scores.
print("Training scores")
print(f"Accuracy:  {train_results['accuracy']:.2%}")
print(f"Precision: {train_results['precision']:.2%}")
print(f"Recall:    {train_results['recall']:.2%}")
print(f"F1-score:  {train_results['f1']:.2%}")

print("\nTest scores")
print(f"Accuracy:  {test_results['accuracy']:.2%}")
print(f"Precision: {test_results['precision']:.2%}")
print(f"Recall:    {test_results['recall']:.2%}")
print(f"F1-score:  {test_results['f1']:.2%}")

Training scores
Accuracy:  97.00%
Precision: 98.62%
Recall:    95.98%
F1-score:  97.29%

Test scores
Accuracy:  94.17%
Precision: 96.77%
Recall:    92.31%
F1-score:  94.49%


## Step 14: Print the Confusion Matrix and Classification Report

The confusion matrix helps us see the types of prediction errors.

The classification report gives a per-class summary of precision, recall, and F1-score.

In [14]:
# Step 14.1: Print the confusion matrix for the test set.
print("Test confusion matrix")
print(test_results['confusion_matrix'])

# Step 14.2: Print the classification report for the test set.
print("\nTest classification report")
print(test_results['classification_report'])

Test confusion matrix
[[53  2]
 [ 5 60]]

Test classification report
              precision    recall  f1-score   support

           0       0.91      0.96      0.94        55
           1       0.97      0.92      0.94        65

    accuracy                           0.94       120
   macro avg       0.94      0.94      0.94       120
weighted avg       0.94      0.94      0.94       120



## Step 15: Inspect a Few Predictions

Finally, we inspect a few individual predictions from the test set.

In [15]:
# Step 15: Show a few example predictions.
for index in range(5):
    print(
        f"{pretty_feature_row(X_test[index])}"
        f" -> predicted={test_predictions[index]}, actual={y_test[index]}"
    )

temp=0.74, occupancy=0.09, business_hours=1, weekday=0, interaction=0.07 -> predicted=0, actual=0
temp=0.45, occupancy=0.58, business_hours=0, weekday=1, interaction=0.26 -> predicted=0, actual=1
temp=0.41, occupancy=0.83, business_hours=1, weekday=0, interaction=0.34 -> predicted=1, actual=1
temp=0.54, occupancy=0.76, business_hours=1, weekday=1, interaction=0.41 -> predicted=1, actual=1
temp=0.30, occupancy=0.27, business_hours=1, weekday=1, interaction=0.08 -> predicted=0, actual=0
